In [1]:
import pandas as pd

vcf = pd.read_csv("gwas_population.vcf", sep="\t")
vcf = vcf.drop(columns=["#CHROM", "POS", "ID", "REF", "ALT", "QUAL"])
vcf = vcf.values.tolist()
vcf = [[list(map(int, elem.split('|'))) for elem in row] for row in vcf]

In [2]:
N_SNPs = len(vcf)
print("Number of SNPs:", N_SNPs)

N = len(vcf[0])
print("Number of Individuals in the Population:", N)

Number of SNPs: 10000
Number of Individuals in the Population: 1000


In [3]:
phenotypes = pd.read_csv("gwas_phenotypes.txt", sep="\t", header=None)
phenotypes.reset_index(drop=True, inplace=True)

# phenotype.drop(columns=[""])
phenotypes = phenotypes.values
phenotype_binary = [i[1] for i in phenotypes]
phenotypes = phenotype_binary
assert len(phenotypes) == N

In [4]:
# covert [1, 1] to 2, [1, 0] and [0, 1] to 1 and [0, 0] to 0 
data = []
for snp in vcf:
    simplified_snp = []
    for i in snp:
        if i == [0, 0]:
            simplified_snp.append(0)
        elif i == [1, 0] or i == [0, 1]:
            simplified_snp.append(1)
        elif i == [1, 1]:
            simplified_snp.append(2)
        else:
            raise AssertionError("Not possible")
    data.append(simplified_snp)

assert len(data) == N_SNPs 
assert len(data[0]) == N

In [5]:
import numpy as np
from scipy.stats import chi2_contingency

pvals = []
# for each SNP have to create a table of
# size 2 columns (disease/healthy) x 3 rows (Genotype1, Genotype2, Genotype3)
for snp in data:
    contingency_table = np.zeros((3, 2))
    for genotype, phenotype in zip(snp, phenotypes):    
        contingency_table[genotype, phenotype] += 1
    contingency_table[contingency_table == 0] = 1e-10
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    pvals.append(p)

In [6]:
significant_snps_count = sum(p < 0.05 for p in pvals)

print(f"Number of SNPs with uncorrected p-value < 0.05: {significant_snps_count}")

Number of SNPs with uncorrected p-value < 0.05: 454


In [7]:
# Calculate corrected p-values using Bonferroni correction
corrected_pvals = [min(p * N_SNPs, 1.0) for p in pvals]  

# Identify SNPs with corrected p-value < 0.05
significant_snps = [(i, pvals[i], corrected_pvals[i]) for i in range(len(corrected_pvals)) if corrected_pvals[i] < 0.05]

significant_snps_df = pd.DataFrame(significant_snps, columns=['SNP Index', 'Uncorrected p-value', 'Corrected p-value'])

# Print the table of significant SNPs and their corrected p-values
significant_snps_df


,SNP Index,Uncorrected p-value,Corrected p-value
0,1000,1.732482e-16,1.732482e-12
1,2000,9.230119e-07,9.230119e-03
2,3000,5.629213e-10,5.629213e-06
3,4000,4.770372e-07,4.770372e-03


In [8]:
odd_ratios_het = []
odd_ratios_homo_alt = []

for index, _, _ in significant_snps:
    snp = data[index]

    homo_ref_disease = 0
    homo_ref_healthy = 0
    heterozygous_disease = 0
    heterozygous_healthy = 0
    homozygous_minor_disease = 0
    homozygous_minor_healthy = 0

    for genotype, phenotype in zip(snp, phenotypes):
        if genotype == 0: 
            if phenotype == 1:  
                homo_ref_disease += 1
            else:  # Healthy
                homo_ref_healthy += 1
        elif genotype == 1:  
            if phenotype == 1:  
                heterozygous_disease += 1
            else:  
                heterozygous_healthy += 1
        elif genotype == 2:  
            if phenotype == 1:  
                homozygous_minor_disease += 1
            else:  
                homozygous_minor_healthy += 1
        
    pr_disease_homo_ref = homo_ref_disease / (homo_ref_disease + homo_ref_healthy)
    pr_disease_heterozygous = heterozygous_disease / (heterozygous_disease + heterozygous_healthy) 
    pr_disease_homozygous_minor = homozygous_minor_disease / (homozygous_minor_disease + homozygous_minor_healthy)


    odds_heterozygous = pr_disease_heterozygous / pr_disease_homo_ref 
    odds_homozygous_minor = pr_disease_homozygous_minor / pr_disease_homo_ref 

    odd_ratios_het.append(odds_heterozygous)
    odd_ratios_homo_alt.append(odds_homozygous_minor)

In [9]:
significant_snps_df["Odds ratio het"] = odd_ratios_het
significant_snps_df["Odds ratio homo alt"] = odd_ratios_homo_alt
significant_snps_df

,SNP Index,Uncorrected p-value,Corrected p-value,Odds ratio het,Odds ratio homo alt
0,1000,1.732482e-16,1.732482e-12,1.582846,2.509911
1,2000,9.230119e-07,9.230119e-03,1.503494,2.248627
2,3000,5.629213e-10,5.629213e-06,1.587617,2.027660
3,4000,4.770372e-07,4.770372e-03,1.344102,1.996176
